# 1. Import libraries and data

In [ ]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from wifiplotting import plot_wifi_heatmap

In [ ]:
wifi_df = pd.read_csv('../data/df_clean.csv')

In [ ]:
ASOF_COORDINATE_COLUMNS = [
    "phone_gps_asof_timestamp",
    "phone_gps_asof_sample_index",
    "latitude", "longitude",
    # "phone_gps_asof_latitude",
    # "phone_gps_asof_longitude",
    "phone_gps_asof_altitude_m",
]
phone_location_df = pd.read_csv("../data/phone_location_log.csv",
                                header=None,
                                names=ASOF_COORDINATE_COLUMNS,
                                na_values=["null", "none", "nan"],
                                keep_default_na=True,)

In [ ]:
wifi_df.isna().sum(axis=0)

In [ ]:
# filter for data in sequoia area

tl_corner = [37.430582, -122.173904]
br_corner = [37.42705, -122.169413]
sequoia = lambda df: (br_corner[0] <= df['latitude']) &  (df['latitude'] <= tl_corner[0]) & \
            (tl_corner[1] <= df['longitude']) & (df['longitude'] <= br_corner[1])

wifi_df = wifi_df[sequoia]
# wifi_df = wifi_df[wifi_df['indoor'] == True]

wifi_df.to_csv('../data/sequoia.csv', index=False, na_rep="null")

In [ ]:
len(wifi_df)

In [ ]:
wifi_df.ap.nunique()

In [ ]:
## plot corner points to see where to sample
# wifi_df.loc[len(wifi_df)] = [np.nan, np.nan, tl_corner[0], tl_corner[1], False, 0, 0, 0, True, np.nan]
# wifi_df.loc[len(wifi_df)] = [np.nan, np.nan, br_corner[0], br_corner[1], False, 0, 0, 0, True, np.nan]

# 2. Some exploratory plots

In [ ]:
wifi_df.head()

In [ ]:
fig, ax, rssi_points, na_points = plot_wifi_heatmap(wifi_df, show_na=0.5)
rows_used = int(rssi_points["sample_count"].sum() + na_points["sample_count"].sum())
print(f"RSSI heatmap uses {len(rssi_points) + len(na_points)} unique coordinate bins from {rows_used} rows.")
plt.show()

In [ ]:
fig, ax, noise_points, na_points = plot_wifi_heatmap(wifi_df, value="noise", invert_cmap=True)
rows_used = int(noise_points["sample_count"].sum())
print(f"Noise heatmap uses {len(noise_points)} unique coordinate bins from {rows_used} rows.")
plt.show()

In [ ]:
snr = wifi_df.rssi - wifi_df.noise

In [ ]:
fig, ax, snr_points, na_points = plot_wifi_heatmap(wifi_df, new_data=snr, plotname='SNR')
rows_used = int(snr_points["sample_count"].sum())
print(f"SNR heatmap uses {len(snr_points)} unique coordinate bins from {rows_used} rows.")
plt.show()

# 3. Indoors vs. outdoors

In [ ]:
sns.histplot(data=wifi_df, x="rssi", hue="indoor", bins=30, kde=True, alpha=0.3)
plt.title("RSSI Distribution (Indoor vs Outdoor)")
plt.show()

# 4. Time of day

Main idea: we know that we should model location and indoor/outdoor, but should we model time? We check by doing simple KNN for location and indoor/outdoor, then seeing if the residuals have any relation to time of day.

Potential problem: because of sampling design, it is possible location and time are correlated.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, Normalizer

In [ ]:
wifi_df.head(5)

In [ ]:
numeric_to_scale = ["latitude", "longitude"]   # columns that need normalization

preprocessor = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), numeric_to_scale),
    ]
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", KNeighborsRegressor())
])

param_grid = {
    "model__n_neighbors": np.arange(10, 41),
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]  # Manhattan vs Euclidean
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",  # or "r2"
    n_jobs=-1,   # use all CPU cores
    verbose=0
)

In [ ]:
var = 'rssi' # 'noise'

df_knn = wifi_df[wifi_df[var].notna()].copy()
# aggregate duplicate X values
# note: this may no longer be necessary after aggregating in preprocessing
df_knn = df_knn.groupby(["latitude", "longitude", "indoor"], as_index=False)[var].mean()
grid.fit(df_knn[["latitude", "longitude", "indoor"]], df_knn[var])

In [ ]:
print("Best params:", grid.best_params_)
print("Best score:", grid.best_score_)

In [ ]:
residuals = wifi_df[var] - grid.best_estimator_.predict(wifi_df[["latitude", "longitude", "indoor"]])
time_sec = wifi_df["time_pdt"].apply(lambda t: int((hms := t.split(':'))[0])*3600 + int(hms[1])*60 + int(hms[2]))

We now visualize residuals vs. time.

In [ ]:
mask = residuals.notna()

In [ ]:
plt.scatter(time_sec[mask], residuals[mask], alpha=0.3)
plt.xlabel('Time of Day (sec)')
plt.ylabel('Residual')
plt.title('Effect of time after KNN on location + indoor/outdoor')
plt.tight_layout()

There doesn't seem to be any meaningful relationship. We will check with a linear regression.

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lr = LinearRegression()
lr.fit(time_sec[mask].values.reshape(-1,1), residuals[mask])

In [ ]:
print(f"Fitted Intercept: {lr.intercept_}")
print(f"Fitted Slope: {lr.coef_[0]}")

# 5. Noise distribution

In [ ]:
plt.hist(residuals[mask], bins=30, density=True, alpha=0.7)

norm_approx = sp.stats.norm(loc=np.median(residuals[mask]), scale=residuals[mask].std()-1)
lap_approx = sp.stats.laplace(loc=np.median(residuals[mask]), scale=residuals[mask].std()-1)
density_grid = np.arange(residuals[mask].min()-1, residuals[mask].max()+1, 1e-3)
plt.plot(density_grid, norm_approx.pdf(density_grid), c='green')
# plt.plot(density_grid, lap_approx.pdf(density_grid), c='red')

plt.xlabel('Residuals')
plt.ylabel('Density')
plt.title('Distribution of residuals after KNN on location + indoor/outdoor')
plt.tight_layout()

# 6. Model Ideas

<div style="display:none">
$
\newcommand{\rssi}{\texttt{RSSI}}
\newcommand{\GP}{\mathcal{GP}}
\newcommand{\K}{\mathcal{K}}
\newcommand{\R}{\mathbb{R}}
\newcommand{\N}{\mathcal{N}}
$
</div>

## Setup

Define the variables of interest.
$$r = \rssi, \quad x = \texttt{longitude}, \quad y = \texttt{latitude}, \quad z = \texttt{indoor}$$

Assume that the signal field is described by a latent function $f$.
$$r_i = f(x_i, y_i, z_i) + \epsilon_i$$
where $\epsilon_i \sim \N(0, \sigma^2_i)$. How should we estimate the noise variance? Do we want to also model variance as a function of $x_i, y_i, z_i$?

<br><br>

## Model

We put a Gaussian process (GP) prior on $f$.
$$f \sim \GP(m, \K)$$
where $m: \R^3 \to \R$ mean function, $\K: \R^3 \times \R^3 \to \R$ covariance kernel. How to set $m$ and $\K$? We see that changes in $z$ lead to much sharper changes in $r$ compared to changes in $x$ and $y$. Can we capture this in our kernel?

<br>

Then, we have the full model below.
$$f \sim \GP(m, \K)$$
$$\vec{r} \mid f \sim \N(f(\vec{x}, \vec{y}, \vec{z}), \Sigma_0)$$
$$\Sigma_0 = \text{noise covariance matrix } \textrm{diag}(\sigma^2_i)$$
Note that under independent Gaussian noise with known/fixed variance, the posterior for this model is available in closed form. If we have unknown variance but homoskedasticity, a simple MCMC solution is available.

<br><br>

## More Advanced Directions

- Our measurements for $x_i$ and $y_i$ are noisy; how can we account for this in our model?
- A refined approach is to account for the WiFi access points on campus, for which we also have data. A reasonable assumption is that there is an access point in each building, though we could treat their locations as unknown parameters as well. Then, there are multiple ways to incoporate this into our model.

  The following are possible ammendments to the model.

  Global GP:
  - Let $d_i$ be the distance of point $(x_i, y_i)$ to its access point. Account for $d_i$ in $m$ and $\K$.
  - Account for access point $a_j$ in $m$ and $\K$. For new observations, set $\hat{f}(x_{new}, y_{new}, z_{new}) = \max_j f(x_{new}, y_{new}, z_{new}, a_j)$. Can also model access point as a random parameter, inducing a hierarchical structure.
  - Condition on additional constraints on the GP dynamics to reflect behavior (e.g. point repellers).

  <br>
  
  Local GPs: For each access point $a_j$, set a different GP prior $f_j \sim \GP(m, \K)$.
  - Define $\displaystyle f(x_i, y_i, z_i) = \sum_j \mathbb{1}\{j = \arg\min_j d(i, j)\} f_j(x_i, y_i, z_i)$, or $\displaystyle f(x_i, y_i, z_i) = \max_j f_j(x_i, y_i, z_i)$

 
    Hard assignments are problematic due to discontinuities on boundaries. Also, location and inferred strength don't necessarily directly correspond with access point due to potential noise and the tendency of devices to "stick" with suboptimal access points until connection becomes too bad.
  
  - Define $\displaystyle f(x_i, y_i, z_i) = \sum_j w_j(x_i, y_i) f_j(x_i, y_i, z_i)$

 
    where $w_j$ is a weight function for access point $j$ based on distance. A mixture reflects the "stickiness."

# 7. Access Points?

In [ ]:
wifi_df['ap'] = wifi_df['ap'].astype('category').cat.codes
fig, ax, noise_points, na_points = plot_wifi_heatmap(wifi_df, value="ap", plotname="Access Points", cmap=plt.colormaps["turbo"])
rows_used = int(noise_points["sample_count"].sum())
print(f"Noise heatmap uses {len(noise_points)} unique coordinate bins from {rows_used} rows.")
plt.show()